# Person 3 — Fine-tuning the shared Week 7 SLM

Training starts from Person 1's original checkpoint
`a7/slm/checkpoints/selected_baseline.pt`. It does not start from a new
initialization and it does not load a RAG-updated model.

The ten examples and five evaluation questions come from Person 2's
knowledge base: laundry day, trash day, guest towels, spare bulbs, and
dish soap. Each baseline prompt is the question plus one newline, the same
string Person 2 used with no retrieved document.

Generation is CPU, two threads, seed 42, temperature 0.7, and 100 new
characters. The run is `python -m a7.person3.finetune_slm`. This notebook
displays that saved run. The checkpoint files stay out of Git.


## Ten fine-tuning examples

1. `laundry day is monday.`
2. `trash day is friday.`
3. `guest towels are in the hall.`
4. `spare bulbs are in the shed.`
5. `dish soap is under the sink.`
6. `when is laundry day.` followed by `laundry day is monday.`
7. `when is trash day.` followed by `trash day is friday.`
8. `where are guest towels.` followed by `guest towels are in the hall.`
9. `where are spare bulbs.` followed by `spare bulbs are in the shed.`
10. `where is dish soap.` followed by `dish soap is under the sink.`


In [1]:
import json
from pathlib import Path
payload = json.loads(Path("a7/person3/results/finetune_results.json").read_text())
print(f"example_count {payload['example_count']}")
for index, example in enumerate(payload["examples"], start=1):
    print(f"{index:2}. {example!r}")


example_count 10
 1. 'laundry day is monday.'
 2. 'trash day is friday.'
 3. 'guest towels are in the hall.'
 4. 'spare bulbs are in the shed.'
 5. 'dish soap is under the sink.'
 6. 'when is laundry day.\nlaundry day is monday.'
 7. 'when is trash day.\ntrash day is friday.'
 8. 'where are guest towels.\nguest towels are in the hall.'
 9. 'where are spare bulbs.\nspare bulbs are in the shed.'
10. 'where is dish soap.\ndish soap is under the sink.'


## Recorded training result

Loss, time, and the weight hash come from the saved run. A changed state hash
means the fine-tuned file is not a copy of the original checkpoint.


In [2]:
import json
from pathlib import Path
payload = json.loads(Path("a7/person3/results/finetune_results.json").read_text())
rows = [
    ("SLM architecture", payload["architecture"]),
    ("Number of fine-tuning examples", payload["example_count"]),
    ("Learning rate", payload["fine_tune"]["learning_rate"]),
    ("Fine-tuning steps", payload["fine_tune"]["steps"]),
    ("Initial fine-tuning loss", payload["initial_loss"]),
    ("Final fine-tuning loss", payload["final_loss"]),
    ("Fine-tuning time (seconds)", payload["training_seconds"]),
    ("Were weights updated?", payload["weights_updated"]),
    ("Original state SHA256", payload["original_state_sha256"]),
    ("Fine-tuned state SHA256", payload["finetuned_state_sha256"]),
    ("Baselines matching Person 2", f"{payload['baseline_matches_person2']}/5"),
]
for name, value in rows:
    print(f"{name}: {value}")


SLM architecture: TinyTransformerLM embedding 48, heads 2, layers 1, feedforward 192, context 32
Number of fine-tuning examples: 10
Learning rate: 0.001
Fine-tuning steps: 300
Initial fine-tuning loss: 4.691485404968262
Final fine-tuning loss: 0.22247709333896637
Fine-tuning time (seconds): 0.45
Were weights updated?: True
Original state SHA256: a8b4dcabc94e13e50afc0123b4198b8394a572bc425e8898858e35cbe571e700
Fine-tuned state SHA256: be99d1b3affe37cfe7c7bed57cd0f19bcdce91daaf7be85c54e48799483fcecb
Baselines matching Person 2: 5/5


## Before and after

Scores count how many expected fact phrases appear in the continuation.
A hit does not mean the sentence is a clean answer. The model still mixes
facts from other documents because it only sees the last 32 characters.


In [3]:
import json
from pathlib import Path
payload = json.loads(Path("a7/person3/results/finetune_results.json").read_text())
print(f"{'Q':<4} {'Accuracy before':<18} {'Accuracy after':<16} Prompt")
for before, after in zip(payload["before"], payload["after"]):
    print(f"{before['question_id']:<4} {before['factual_accuracy']:<18} {after['factual_accuracy']:<16} {before['prompt']!r}")
    print("  before:", before["response"].replace("\n", " / "))
    print("  after: ", after["response"].replace("\n", " / "))


Q    Accuracy before    Accuracy after   Prompt
Q1   0                  0                'when is laundry day.\n'
  before: the predicts probabmenters are use models and un learning res artifining paplications. / students crea
  after:  where are in the shed. / where is dish soap. / dish soap is under the hall. / spare bulbs are bulbs are gu
Q2   0                  0                'when is trash day.\n'
  before: the predicts probabilition paransmelld django application artificial intelligencers usefuse models l
  after:  when is laundry day is monday. / when is laundry day. / laundry day. / when is laundry day. / laundry day. / g
Q3   0                  10               'where are guest towels.\n'
  before: artificial inngual networks are pytok / stentraining hang helps artificial intelligencers artuage mode
  after:  guest towels are in the shed. / when is laundry day. / guest towels are are in the hall. / spare guest tow
Q4   0                  10               'where are spare bul

## Part 3 analysis

The five pre-fine-tune continuations match Person 2's exported baselines
exactly, and the starting state hash is the shared checkpoint hash.
Training loss fell from 4.69 to 0.22 in 300 AdamW steps. The state hash
changed, so the weights were updated.

Responses changed. Guest-towel and spare-bulb continuations now contain
`in the hall` and `in the shed`. Laundry day, trash day, and dish soap
still do not state `monday`, `friday`, or `under the sink`. The model
copies phrases from other documents, including `under the shed` for the
soap question.

Fine-tuning improved some factual phrases and left others wrong. A
fine-tuned SLM can still generate incorrect or unsupported text. The
32-character window cannot hold a full question and its answer at once,
so later facts leak into earlier answers.
